# Pattern Detection – Day 10: Analysis & Critical Reflection

**Student Name:** Traver Dinten
**Country:** Switzerland
**Semester term:** FS26

This notebook presents the analytical discussion and critical reflection for the Canny edge detection evaluation on Sentinel-2 imagery of the Aletsch Glacier tongue. No new simulations or computations are introduced. All arguments refer to the quantitative results established in Day 9 (Edge Pixel Density evaluation) and the experimental design defined in Day 8.

## Observations

The Edge Pixel Density (EPD) varies dramatically across configurations. In the **blur sweep** (thresholds fixed at 50/150), only two of four configurations produce non-zero edge detections: $\sigma=0.5$ yields EPD = 0.0110 (+321% relative to baseline), while the baseline $\sigma=1.0$ yields EPD = 0.0026. Both $\sigma=2.0$ and $\sigma=4.0$ produce EPD = 0.0000, i.e., zero detected edge pixels.

In the **threshold sweep** ($\sigma$ fixed at 1.0), the pattern is analogous: thresholds (30/90) yield EPD = 0.0160 (+516% vs. baseline), the baseline (50/150) yields EPD = 0.0026, and both (80/200) and (100/250) yield EPD = 0.0000.

Thus, only 3 of the 7 unique configurations produce any detectable edges. The transition from detectable to zero edges is abrupt — there is no gradual degradation.

## Interpretation

The abrupt collapse of edge detections is a direct consequence of the image's intensity characteristics. The `load_image` pipeline rescales Sentinel-2 surface reflectance values (0–10,000) to `uint8` (0–255), compressing the gradient magnitudes between glacier, moraine, and rock surfaces to a narrow band of pixel-value differences. The Canny algorithm's gradient magnitudes are therefore inherently small in absolute terms.

**Blur effect:** At $\sigma \geq 2.0$, the Gaussian pre-filtering smooths these already-weak intensity transitions below the fixed hysteresis thresholds (50/150), eliminating all edge responses. The 10 m spatial resolution of Sentinel-2 means that structural boundaries (medial moraines, lateral ice-rock contacts) span only a few pixels, making them highly susceptible to over-smoothing.

**Threshold effect:** At thresholds $\geq$ (80/200), the required gradient magnitudes exceed what the rescaled image can physically produce. The constant 1:3 ratio between $T_{low}$ and $T_{high}$ ensures that even weak edges connected to strong edges cannot survive, because no gradients reach $T_{high}$.

Conversely, the most sensitive configurations ($\sigma=0.5$ or thresholds 30/90) detect 4–6× more edges than the baseline. However, visual inspection from Day 8 suggests that much of this increase consists of texture responses from crevasse fields and uneven snow/ice surfaces rather than the targeted structural boundaries.

## Discussion and Critical Reflection

**Practical applicability:** The baseline configuration ($\sigma=1.0$, thresholds 50/150) represents the only viable operating point for this use case. It detects some structural boundaries while suppressing the majority of surface texture. However, with an EPD of only 0.26%, the detected edges are sparse and may not form continuous boundary lines — a critical requirement for moraine delineation or terminus tracking.

**Algorithm suitability:** The Canny algorithm, while theoretically appropriate for detecting linear boundaries, proves highly sensitive to the specific intensity characteristics of satellite imagery. The narrow dynamic range after `uint8` rescaling limits the effective parameter space to a small window. Four of seven configurations yield zero output, providing no information for comparison. This binary behavior (either some detections or none) undermines the goal of systematic parameter optimization.

**Limitations of EPD:** The Edge Pixel Density metric captures detection sensitivity but does not distinguish between true structural boundaries and noise. A configuration with higher EPD is not necessarily better — it may simply detect more surface texture. A more informative evaluation would require ground-truth annotations or a complementary metric such as edge continuity or structural coherence.

**Risks and improvements:** The primary risk is that the `uint8` quantization step discards gradient information critical for Canny's hysteresis thresholding. Operating directly on the original `float64` reflectance values (or at minimum `uint16`) and recalibrating thresholds to the image's actual gradient distribution would likely expand the useful parameter range and improve boundary detection quality. Additionally, adaptive thresholding based on local gradient statistics could address the spatially varying contrast between glacier zones.

In conclusion, while Canny edge detection can identify some structural features in Sentinel-2 Aletsch Glacier imagery, its practical utility for automated moraine delineation is limited by the image's narrow dynamic range and the algorithm's sensitivity to preprocessing choices.